In [ ]:
import csv

# 1. CARICAMENTO DATI

nome_file = "formula1_data.csv"

# Apro il file in modalità lettura (default) assicurandomi di leggere bene gli accenti
with open(nome_file, encoding="utf-8") as file:
    dataset_f1 = list(csv.DictReader(file))

# Mappatura dei punti secondo il regolamento ufficiale del 2008
punti_2008 = {"1": 10, "2": 8, "3": 6, "4": 5, "5": 4, "6": 3, "7": 2, "8": 1}


# 2. FUNZIONI DI ANALISI

def analizza_performance_pilota(dati, nome_pilota):
    """Calcola le statistiche base (punti, vittorie, podi) di un singolo pilota."""
    punti_totali = 0
    vittorie = 0
    podi = 0

    for riga in dati:
        # Uso .lower() per rendere la ricerca insensibile a maiuscole/minuscole
        if riga['Driver'].lower() == nome_pilota.lower():
            pos = riga['Position']

            # Se la posizione è tra i primi 8, aggiungo i punti corrispondenti
            if pos in punti_2008:
                punti_totali += punti_2008[pos]

            # Incremento il contatore delle vittorie
            if pos == "1":
                vittorie += 1

            # Incremento il contatore dei podi (prime 3 posizioni)
            if pos == "1" or pos == "2" or pos == "3":
                podi += 1

    return [punti_totali, vittorie, podi]


def genera_classifica_piloti(dati):
    """Crea la classifica generale dei piloti e la salva in un file di testo."""
    classifica = {}

    for riga in dati:
        pilota = riga['Driver']
        pos = riga['Position']

        # Recupero i punti della singola gara (0 se fuori dalla zona punti)
        punti_gara = 0
        if pos in punti_2008:
            punti_gara = punti_2008[pos]

        # Logica di accumulo: sommo i punti se il pilota esiste già, altrimenti lo creo
        if pilota in classifica:
            classifica[pilota] += punti_gara
        else:
            classifica[pilota] = punti_gara

    # Ordino il dizionario in base ai punti totali (dal maggiore al minore)
    classifica_ordinata = dict(sorted(classifica.items(), key=lambda x: x[1], reverse=True))

    # Genero il report testuale richiesto dalle specifiche del progetto
    with open("Drivers_Standings_2008.txt", "w", encoding="utf-8") as f:
        f.write("Drivers Standings 2008 Formula 1\n")
        for pilota, punteggio in classifica_ordinata.items():
            f.write(f"{pilota}: {punteggio}\n")

    return classifica_ordinata


def genera_classifica_costruttori(dati):
    """Calcola i punti totali per ogni scuderia sommando i risultati delle vetture."""
    classifica_team = {}

    # Stessa logica della classifica piloti, ma raggruppando per la colonna 'Team'
    for riga in dati:
        team = riga['Team']
        pos = riga['Position']

        punti_gara = 0
        if pos in punti_2008:
            punti_gara = punti_2008[pos]

        if team in classifica_team:
            classifica_team[team] += punti_gara
        else:
            classifica_team[team] = punti_gara

    classifica_ordinata = dict(sorted(classifica_team.items(), key=lambda x: x[1], reverse=True))
    return classifica_ordinata


# 3. ESECUZIONE DEL PROGRAMMA

print("Analisi F1 2008")

# Estraggo un elenco univoco dei piloti per facilitare la ricerca all'utente
piloti_disponibili = []

for riga in dataset_f1:
    pilota = riga['Driver']
    if pilota not in piloti_disponibili:
        piloti_disponibili.append(pilota)

# Metto la lista in ordine alfabetico per una migliore leggibilità
piloti_disponibili.sort()

print("\nElenco dei piloti che puoi cercare:")
print(piloti_disponibili)
print("\n")


# Gestione sicura dell'input utente
nome_valido = False

# Il ciclo continua a chiedere il nome finché non viene inserito un pilota esistente
while nome_valido == False:
    pilota_scelto = input("Inserisci il cognome del pilota: ")
    pilota_scelto = pilota_scelto.strip() # Rimuovo spazi vuoti accidentali

    for p in piloti_disponibili:
        if pilota_scelto.lower() == p.lower():
            nome_valido = True

    if nome_valido == False:
        print("Pilota non trovato! Controlla l'elenco e riprova.\n")


# Stampa dei risultati finali
risultati_pilota = analizza_performance_pilota(dataset_f1, pilota_scelto)

print(f"\nRISULTATI {pilota_scelto.upper()}")
print(f"Punti totali: {risultati_pilota[0]}")
print(f"Vittorie: {risultati_pilota[1]}")
print(f"Podi: {risultati_pilota[2]}")


print("\nCLASSIFICA PILOTI\n")
classifica_p = genera_classifica_piloti(dataset_f1)
for p, pt in classifica_p.items():
    print(f"{p}: {pt} pt")


print("\nCLASSIFICA COSTRUTTORI\n")
classifica_c = genera_classifica_costruttori(dataset_f1)
for t, pt in classifica_c.items():
    print(f"{t}: {pt} pt")

Analisi F1 2008

Elenco dei piloti che puoi cercare:
['Alonso', 'Glock', 'Hamilton', 'Heidfeld', 'Kovalainen', 'Kubica', 'Massa', 'Raikkonen', 'Trulli', 'Vettel']


Inserisci il cognome del pilota: c
Pilota non trovato! Controlla l'elenco e riprova.

Inserisci il cognome del pilota: massa

RISULTATI MASSA
Punti totali: 97
Vittorie: 6
Podi: 10

CLASSIFICA PILOTI

Hamilton: 98 pt
Massa: 97 pt
Raikkonen: 75 pt
Kubica: 75 pt
Alonso: 61 pt
Heidfeld: 60 pt
Kovalainen: 53 pt
Vettel: 35 pt
Trulli: 31 pt
Glock: 25 pt

CLASSIFICA COSTRUTTORI

Ferrari: 172 pt
McLaren: 151 pt
BMW: 135 pt
Renault: 61 pt
Toyota: 56 pt
Toro Rosso: 35 pt
